In [82]:
%run Hilbert_C^2_package.ipynb

# generate all generalized partition with length k, positive part length l, positive part partition of n
genpars = lambda k,l,n:{i:x for i,x in \
enumerate(list(itertools.product(Partitions(n+1,length = k-l).list(), Partitions(n, length = l).list())))}

In [ ]:
import math, time

# compute f_K(\lambda) for generalized partition of the form (-n-1,x1,x2,...,xk)
# this is function defined in LQW's 'Hilbert schemes and W algebra'
start_time = time.time()

# select a pair of indeces between 1 and n igoring the order of arrangement.
pair = lambda n:[(i, j) for i in range(0, n) for j in range(i+1, n)]

# compute up to (m-1)-th f_Kp
scale = 18

# initialize the setting
F = FormalVariables(scale)

# f_K for single variable. p stands for positive
f0 = lambda first:-first[0]
f_Kp = [f0(F.variables())]

# the recursion process
for n in range(2,scale):

    # comes from iteration formula
    const = -1/2*(-1)^n*math.factorial(n)*(F.sqsum(n) - (F.l1sum(n) + 1)^2 - 1)
    itera1 = -sum([(F.x(p[0]) + F.x(p[1]))*f_Kp[n-2](F.trans(p[0],p[1],n)) for p in pair(n)])
    itera2 = sum([(sum(F.rmv(i,n)) + 1)*f_Kp[n-2](F.rmv(i,n)) for i in range(0,n)])

    # f_K of next degree
    f = const + itera1 + itera2
    
    # add into the list
    f_Kp.append(f)

# show time consumed by this code
end_time = time.time()
list_time = end_time - start_time
print(f"Time taken generating up to {scale-1}-th f_Kp: {list_time:.6f} seconds")

In [101]:
#########################################
# save the result into json format file #
#########################################
%%time
import json

# convert f_K into dictionary with degree as keys, coeffienct as values
f_Kp_dict = [{str(key):str(value) for key, value in mpoly.dict().items()} for mpoly in f_Kp]

# path to data
file_path = "/Users/huajuan/Documents/computation of f_K/f_K.json"

# save the data
with open(file_path, 'w') as file:
    json.dump(f_Kp_dict, file)
    file.close()
print('Done')

Done


In [102]:
#############################
# load the data into a list #
#############################
%%time
import json

# path to data
file_path = "/Users/huajuan/Documents/computation of f_K/f_K.json"

# load the data
with open(file_path, 'r') as file:
    data = json.load(file)
    file.close()
    
####################
# Process the data #
####################

# extract the number of variables
length = len(eval(list(data[0].keys())[0]))
R = FormalVariables(length)

# recover key and value into tuple and int resp.
recovered_data = [{eval(key):eval(value) for key, value in mpoly_dict.items()} for mpoly_dict in data]

# load the dict of mpoly back into multivariable polynomials
f_Kp_reload = [R.load_dict(mpoly_dict) for mpoly_dict in recovered_data]

In [ ]:
# compute f_K(\lambda) for generalized partition of the general form

# define the list of generalized partition of length k
# the list of all generalized partition has two indeces, 
# one is position part of length l, negative part of length k-l;
# another is that positive part is the partition of n, while negative part is partition of n+1.
# We can call it generalized partition of type (l,n) of length k, where 0<l<k
import itertools, time
import numpy as np



genpars_list = lambda k,l,n:\
list(itertools.product(Partitions(n+1,length = k-l).list(), Partitions(n, length = l).list()))
# a function to search the key of an item 
def find_key_by_value(my_dict, value_to_find):
    try:
        return next(key for key, value in my_dict.items() if value == value_to_find)
    except StopIteration:
        raise ValueError(f'value {value_to_find} not found in dictionary')

# define the induction operation {lambda ---> lambda_i,j}
def stickpar(tup):# ([5,1],[3,2]) ---> [-5,-1,3,2]
    return [-x for x in tup[0]]+[x for x in tup[1]]
def cutpar(lst):
    #l = list(itertools.product([[x for x in list if x<0]],[[x for x in list if x>0]]))
    l1,l2 = [-x for x in lst if x<0],[x for x in lst if x>0]
    l1.sort(reverse=True)
    l2.sort(reverse=True)
    return (Partition(l1),Partition(l2))
def indpar(lst,i,j):
    a = lst[i]+lst[j]
    if lst[i]+lst[j] == 0:
        return tuple([])
    else:
        nlst = [lst[x] for x in range(0,len(lst)) if x!=i]
        nlst = [nlst[x] for x in range(0,len(nlst)) if x!=j-1]
        nlst.append(a)
        return cutpar(nlst)
    
#search the index of a generalized partition
def keypar(par):
    if par!=[]:
        key = find_key_by_value(genpars(len(stickpar(par)),len(par[1]),sum(par[1])),par)
        return key
    else:
        print("None")

#select a pair of indeces between 1 and n igoring the order of arrangement.
pair = lambda n:[(i, j) for i in range(0, n) for j in range(i+1, n)]

    
# we can define the value of f_K to be another dictionary related to generalized partitions by key.
# when doing induction, type (l,n) partition could become type (l-1,n) of length k-1, type (l,n) of length k-1,
    # type (l,n-i^-) of length k-1 and type (l-1,n-i^+) of length k-1
# induction starts from type (1,1) of length 2

pair = lambda n:[(i, j) for i in range(0, n) for j in range(i+1, n)]
# select a pair of indeces between 1 and n igoring the order of arrangement.

###############################################################
###############################################################
N = 22 # set the upper bound of positive part sum to be N-1
K = 7 # set the upper bound of length to be K-1
###############################################################
###############################################################

data_f_K = {}
#print(type(data_f_K))
data_f_K[tuple([])] = 0
start_time = time.time() # measuring the iteration time
for i in range(1,N):
    data_f_K[genpars(2,1,i)[0]] = -genpars(2,1,i)[0][1][0]

# generate value of f_K by induction:
for k in range(3,K):
    fac_k = math.factorial(k-1)*(-1)^k*(1/2)
    pair_k = pair(k)
    for l in range(1,k):
    #for l in range(2,3):
        for n in range (1,N):
            for dex in range(0,len(genpars(k,l,n))):
                par = genpars(k,l,n)[dex]
                data_f_K[par] = fac_k*(sum([x*x for x in par[1]])-sum([x*x for x in par[0]])-1)\
                - sum([(stickpar(par)[p[0]]\
                        +stickpar(par)[p[1]])*data_f_K[indpar(stickpar(par),p[0],p[1])] for p in pair_k])
                #print(par[0])
                #print([data_f_K[indpar(stickpar(par),p[0],p[1])] for p in pair_k])
        
iter_time = time.time() - start_time            
print("iter_time: ",iter_time,"s")

# DO NOT RUN THIS PART!!!
# UNLESS WE NEED TO UPDATE THE DATA


In [ ]:
# Save the data_f_K in json format
import json

# measuring the time of saving data
start_time = time.time()

# Create a new dictionary with string keys
json_dict = {str(key): str(value) for key, value in data_f_K.items()}

# Specify the file path where you want to save the JSON data
file_path = '/Users/huajuan/Documents/computation of f_K/data_f_K.json'

# Open the file in write mode and serialize the dictionary to JSON
with open(file_path, 'w') as json_file:
    json.dump(json_dict, json_file)

iter_time = time.time() - start_time 
print("elapsed_time: ",iter_time,"s")

In [ ]:
# Loading data in data_f_K

import ast
import json
import time

def convert_to_partition_tuple(partition_tuple_str):
    partition_tuple = ()
    for strs in ast.literal_eval(partition_tuple_str):
        partition_tuple += (Partition(strs), )
    return partition_tuple

# measuring the time of loading data
start_time = time.time()

# Specify the file path where the JSON data is stored
file_path = '/Users/huajuan/Documents/computation of f_K/data_f_K.json'

# Read the JSON data from the file
with open(file_path, 'r') as json_file:
    json_dict = json.load(json_file)

# Convert string keys back to tuples if necessary
data_f_K = {}

for key, value in json_dict.items():
    new_key = convert_to_partition_tuple(key)
    data_f_K[new_key] = int(value)
    
# Access and use the loaded dictionary
#print(data_f_K)

iter_time = time.time() - start_time 
print("elapsed_time: ",iter_time,"s")

In [105]:
# compute f_K(\lambda) for generalized partition of the form ([sum(lam),1], lam).

start_time = time.time()

# select a pair of indeces between 1 and n igoring the order of arrangement.
# contain pair of two numbers within 0,...,n-1
pair = lambda n:[(i, j) for i in range(0, n) for j in range(i+1, n)]

# number of variables. It is inherited from that of f_Kp_reload.
scale = len(f_Kp_reload[0].parent().gens())

# compute up to dim-th f_Kp2.
dim = scale - 1

# initialize the setting
F = FormalVariables(scale)

# f_K for single variable. p stands for positive
f0 = lambda first: - 3*first[0] + 3
f_Kp2 = [f0(F.variables())]

# the recursion process
for n in range(1,dim):

    # comes from iteration formula
    const = -1/2*(-1)^n * math.factorial(n+2)*(F.sqsum(n+1) - F.l1sum(n+1)^2 - 2)
    #print('\nconst:',const)
    
    itera1 = sum([(F.x(i)-1)*f_Kp_reload[n](F.minus1(i,n+1)) for i in range(0,n+1)])
    #print('itera1:',itera1)
    
    itera2 = sum([-(F.l1sum(n+1)-F.x(i)) * f_Kp2[n-1](F.rmv(i,n+1)) for i in range(0,n+1)])
    #print('itera2:',itera2)
    
    itera3 = sum([(F.x(p[0])+F.x(p[1]))*f_Kp2[n-1](F.trans(p[0],p[1],n+1)) for p in pair(n+1)])
    #print('itera3:',itera3)
    
    itera4 = -(F.l1sum(n+1)+1)*f_Kp_reload[n]
    #print('itera4:',itera4)
    
    # f_K of next degree
    f = const - itera1 - itera2 - itera3 - itera4 
    #print('result is:', f)
    
    # add into the list
    f_Kp2.append(f)

# show time consumed by this code
end_time = time.time()
list_time = end_time - start_time
print(f"Time taken generating up to {dim}-th f_Kp2: {list_time:.6f} seconds")

Time taken generating up to 14-th f_Kp2: 27.005828 seconds


In [106]:
%%time
# save the result into json format file

file_path = "/Users/huajuan/Documents/computation of f_K/f_Kp2.json"
f_K().save(file_path, f_Kp2)

CPU times: user 210 ms, sys: 7.36 ms, total: 217 ms
Wall time: 217 ms


In [107]:
%%time
# load the data into a list

file_path = "/Users/huajuan/Documents/computation of f_K/f_Kp2.json"
f_Kp2_reload = f_K().load(file_path)

CPU times: user 18.4 s, sys: 58.2 ms, total: 18.5 s
Wall time: 18.5 s


14
